In [17]:
import sqlite3
import pandas as pd
import statsmodels.api as sm
import numpy as np
import functools
import math
from datetime import datetime
from zoneinfo import ZoneInfo

1. 
#1. define database connection
db_address = "C:/Users/USER/Downloads/study docs/Comp604/assignment/assignment2/New Version/database.db"

#2 establish database connection
db_connect = sqlite3.connect(db_address)

#3 pull data out
#3.1 Method 1
#df_products = pd.read_sql_query('SELECT * FROM products', db_connect)
#df_suppliers = pd.read_sql_query('SELECT * FROM suppliers', db_connect)
#df_logistics = pd.read_sql_query('SELECT * FROM logistics', db_connect)
#df_supplychaindatas = pd.read_sql_query('SELECT * FROM supply_chain_datas', db_connect)

#for df in [df_products, df_suppliers, df_logistics, df_supplychaindatas]:
#    if 'index' in df.columns:
#        df.drop(columns="index", inplace=True)
#df_data = pd.merge(df_supplychaindatas, df_products, how="inner", on="SKU")
#df_data2 = pd.merge(df_data, df_suppliers, how="inner", on="Supplier index")
#df_data3 = pd.merge(df_data2, df_logistics, how="inner", on="Logistics index")

#3.2 Method 2
table_names = ['supply_chain_datas', 'products', 'suppliers', 'logistics']
tables = {}
for name in table_names:
    tables[name] = pd.read_sql_query(f"SELECT * FROM {name}", db_connect)
df_data = tables['supply_chain_datas']
df_data2 = tables['products']
db_connect.close()
    
#2. perform statistical test:
#2.1 check if stock levels are above threshold:
required_stocklevel = 20 
#2.1.1 Method 1
print(f"CHECK IF STOCK LEVELS ARE ABOVE THRESHOLD:")
stock_to_check = []
for cell in df_data['Stock levels']:
    if pd.notna(cell):
        if int(float(cell)) < required_stocklevel:
            print(f"Stock of {cell} is low, please reorder")
            stock_to_check.append(cell)
    else: 
        print(f"data for {cell} is blank, please check database")

#2.1.2 Method 2
low_stock = df_data[df_data['Stock levels']<required_stocklevel]
print(low_stock)

#2.2 define priority stocks that need assessment first:
print(f"PRITORITY CHECKING:")
#for cell2 in df_data['Number of products sold']:
#    for cell3 in stock_to_check:
#        if cell2 == cell3 and cell3>500:
#            print(f"Stock of {cell2} is to check in high priority")
# Or use Pandas condition:
priority = df_data[(df_data['Stock levels']<required_stocklevel) & (df_data['Number of products sold']>500)]
print(priority[['SKU', 'Stock levels', 'Number of products sold']])

#2.3 classify stock into categories depending on the Stock levels:
print(f"CLASSIFY STOCK FUNCTION")
def classify_stock(stock):
    if pd.notna(stock):
        if stock<5:
            return "critical"
        elif stock>40:
            return "overstock"
        else:
            return "normal"
#classify_stock(df_data[(df_data['SKU'] == 'SKU5')])
#status = classify_stock(3) 
#print(status)

df_data['Stock status'] = df_data['Stock levels'].apply(classify_stock)
print(df_data[['SKU', 'Stock levels', 'Stock status']])

#2.4 print SKU of products sold that is not in dim table product list:
#2.4.1 Method 1
print(f"print SKU of products sold that is not in dim table product list:".upper())
#df_data_set = set(df_data['SKU'])
#df_data2_set = set(df_data2['SKU'])
#mismatch = df_data_set-df_data2_set
#print(mismatch)

#2.4.2 Method 2
def check_relationship(fact, dimention, key):
    df_data_set = set(fact[key])
    df_data2_set = set(dimention[key])
    mismatch = df_data_set-df_data2_set
    return mismatch

check_relationship(df_data, df_data2, 'SKU')

#2.4.2.1 Define your list of validation tuples based on your assignment prompt
print(f"Define your list of validation tuples based on your assignment prompt".upper())
# Structure: (table_name, target_row_count, description_or_key_column)
validation_tuples = [('supply_chain_datas', 'products', 'SKU'), ('supply_chain_datas', 'suppliers', 'Supplier index'), ('supply_chain_datas', 'logistics', 'Logistics index')]
validation_results = []
for supplychaindatas, dim, key2 in validation_tuples:
    results = check_relationship(tables[supplychaindatas], tables[dim], key2)
    print(results)
    validation_results.append(results)
print(validation_results)

#2.5 Combine revenue into one total:
print(f"Combine revenue into one total".upper())
revenue_values = df_data['Revenue generated'].dropna().tolist()
total = functools.reduce(lambda a, x:a+x, revenue_values, 0)
print(total)

#2.6 Broadcast in Numpy:
print(f"Broadcast in Numpy".upper())
#2.6.1.1 Method 1
#df_data_array = np.arange(len(df_data['Stock levels'])).reshape(len(df_data['Stock levels']), 1)

#2.6.1.2 Method 2
selected_columns = ['Revenue generated', 'Shipping costs', 'Defect rates']

numeric_data = df_data[selected_columns].apply(pd.to_numeric)
numeric_numpy_data = numeric_data.to_numpy()
print(f"Shape: {np.shape(numeric_numpy_data)}")
print(f"Dimentions: {numeric_numpy_data.ndim}")
print(f"Data type: {numeric_numpy_data.dtype}")
print(f"print 5 rows: {numeric_numpy_data[0:5]}")

#2.6.2 Keep rows where all three values are finite:
#apply logic (isfinite) to every single cell of the data numpy array:
finite = np.isfinite(numeric_numpy_data)
print(finite)

#checking (all) the test of {finite} but for a whole group, because axis=1 (horizontall or a whole row) so conditions (isfinite) 
#will be checked for all values in cells belong to 1 whole horizontal row in the matrix: 
finite_across = np.all(finite, axis=1)

#apply masked boolean values dataframe to only those satisfy condition (filtered by finite_across)
finite_data = numeric_numpy_data[finite_across] 
print(finite_data)

#perform calculations on the extracted data table:
print(f"Total Revenue generated: {finite_data[:,0].sum(axis=0)}")
print(f"Mean shipping costs: {finite_data[:,1].mean(axis=0)}")
print(f"Mean defect rates: {finite_data[:,2].mean(axis=0)}")

#2.7 Calculate shipping costs
print(f"Calculate shipping costs".upper())
shipping_costs_clean = df_data['Shipping costs'].apply(pd.to_numeric)
shipping_costs_numpy = shipping_costs_clean.to_numpy()
shipping_costs_finite = np.isfinite(shipping_costs_numpy)
shipping_costs = shipping_costs_numpy[shipping_costs_finite]
multipliers = [1, 1.05, 1.1]
#calculated_scenarios = functools.reduce(lambda x, shipping_costs: shipping_costs*x, multipliers)
#print(calculated_scenarios)
calculated_scenarios = shipping_costs.reshape(-1, 1)*multipliers
print(calculated_scenarios)

#2.8 Warehouse scenario:
#suppose 1 box holds 12 units
print(f"Warehouse scenario: print number of boxes required".upper())
def boxes_required(item_count):
# rounds down, which will be 1 box short
#    boxes = item_count//12
    if pd.isna(item_count):
        return None
    else: 
        boxes = math.ceil(item_count/12)
    return boxes
df_data['Boxes required'] = df_data['Stock levels'].apply(boxes_required)
print(df_data['Boxes required'])
    
#2.9 Record when an analysis occurs
print(f"Record when an analysis occurs".upper())
x = datetime.now()
print(x)

log_message = (
    f"Run time: {x.strftime('%c')}"
    f"Records analysed: {len(df_data)}"
    f"Total recorded revenue: {finite_data[:,0].sum(axis=0)}"
    f"Number of low stock products: {len(low_stock)}"
)

text_log = "C:/Users/USER/Downloads/study docs/Comp604/assignment/assignment2/New Version/log.txt"
with open(text_log, mode="a") as file:
    file.write(log_message)

CHECK IF STOCK LEVELS ARE ABOVE THRESHOLD:
Stock of 1 is low, please reorder
Stock of 5 is low, please reorder
Stock of 11 is low, please reorder
Stock of 5 is low, please reorder
Stock of 14 is low, please reorder
Stock of 9 is low, please reorder
Stock of 2 is low, please reorder
Stock of 10 is low, please reorder
Stock of 4 is low, please reorder
Stock of 6 is low, please reorder
Stock of 4 is low, please reorder
Stock of 1 is low, please reorder
Stock of 18 is low, please reorder
Stock of 13 is low, please reorder
Stock of 4 is low, please reorder
Stock of 5 is low, please reorder
Stock of 10 is low, please reorder
Stock of 13 is low, please reorder
Stock of 12 is low, please reorder
Stock of 0 is low, please reorder
Stock of 10 is low, please reorder
Stock of 17 is low, please reorder
Stock of 16 is low, please reorder
Stock of 5 is low, please reorder
Stock of 15 is low, please reorder
Stock of 5 is low, please reorder
Stock of 15 is low, please reorder
    index Product type    